In [1]:
import os
import gc
import random
import warnings
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torchvision
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import OneCycleLR
from tqdm import tqdm
import albumentations as A
from transformers import (
    DeformableDetrImageProcessor,
    DeformableDetrConfig,
    DeformableDetrForObjectDetection,
)
from coco_eval import CocoEvaluator

warnings.filterwarnings("ignore")
print("Torch version:", torch.__version__)

/home/gb10/miniconda3/envs/cv-hw2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch version: 2.11.0+cu130


In [2]:
# =========================================================
# 1. Basic settings
# =========================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ===== Change to your correct path =====
data_path = "nycu-hw2-data"

TRAIN_DIRECTORY = os.path.join(data_path, "train")
VAL_DIRECTORY = os.path.join(data_path, "valid")

TRAIN_ANNOTATION_FILE = os.path.join(data_path, "train.json")
VAL_ANNOTATION_FILE = os.path.join(data_path, "valid.json")

# ===== checkpoint（ only use backbone pretrained and image processor）=====
CHECKPOINT = "SenseTime/deformable-detr"

# ===== hyperparameter =====
BATCH_SIZE = 8
NUM_EPOCHS = 100
LR = 2e-5
LR_BACKBONE = 2e-6
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
FREEZE_EPOCHS = 5

# ===== Image size =====
DATA_SIZE = 384

# ===== gragient setting ====
USE_AMP = True if DEVICE == "cuda" else False
GRAD_ACCUM_STEPS = 4  # effective batch = 8 x 4 = 32
ENABLE_GRADIENT_CHECKPOINTING = True

# ===== Other setting =====
GRAD_CLIP_NORM = 0.5
SAVE_DIR = "./checkpoints_deformable_detr"
os.makedirs(SAVE_DIR, exist_ok=True)

# ===== Reproducibility and Random seed =====
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Using device: cuda


In [3]:
# =========================================================
# 2. Image processor
# =========================================================
image_processor = DeformableDetrImageProcessor.from_pretrained(
    CHECKPOINT,
    size={"shortest_edge": DATA_SIZE, "longest_edge": DATA_SIZE},
)

print(image_processor)

DeformableDetrImageProcessor {
  "do_convert_annotations": true,
  "do_normalize": true,
  "do_pad": true,
  "do_rescale": true,
  "do_resize": true,
  "format": "coco_detection",
  "image_mean": [
    0.485,
    0.456,
    0.406
  ],
  "image_processor_type": "DeformableDetrImageProcessor",
  "image_std": [
    0.229,
    0.224,
    0.225
  ],
  "resample": 2,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "longest_edge": 384,
    "shortest_edge": 384
  }
}



In [4]:
# =========================================================
# 3. Visualization helpers
# =========================================================
BOX_COLOR = (255, 0, 0)
TEXT_COLOR = (255, 255, 255)


def visualize_bbox(img, bbox, class_name=None, color=BOX_COLOR, thickness=2):
    x, y, w, h = bbox
    x1, y1 = int(round(x)), int(round(y))
    x2, y2 = int(round(x + w)), int(round(y + h))
    cv2.rectangle(img, (x1, y1), (x2, y2), color=color, thickness=thickness)
    if class_name is not None:
        (text_width, text_height), _ = cv2.getTextSize(
            class_name, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1
        )
        cv2.rectangle(
            img, (x1, y1 - int(1.3 * text_height)), (x1 + text_width, y1), color, -1
        )
        cv2.putText(
            img,
            class_name,
            (x1, y1 - 3),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            TEXT_COLOR,
            lineType=cv2.LINE_AA,
        )
    return img


def draw_coco_annotations(image, annotations, id2label=None, title=None):
    img = image.copy()
    for ann in annotations:
        bbox = ann["bbox"]
        category_id = ann["category_id"]
        class_name = (
            str(category_id)
            if id2label is None
            else id2label.get(category_id, str(category_id))
        )
        img = visualize_bbox(img, bbox, class_name=class_name)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    if title is not None:
        plt.title(title)
    plt.axis("off")
    plt.show()

In [5]:
# =========================================================
# 4. Augmentation
# =========================================================
train_transform = A.Compose(
    [
        A.RandomBrightnessContrast(p=0.3),
        A.RandomGamma(p=0.2),
        A.GaussianBlur(blur_limit=(3, 5), p=0.1),
        A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=0.10,
            rotate_limit=3,
            border_mode=0,
            p=0.30,
        ),
        A.CoarseDropout(max_holes=4, max_height=20, max_width=20, p=0.2),
    ],
    bbox_params=A.BboxParams(format="coco", label_fields=["category_ids"]),
)

In [6]:
# =========================================================
# 5. Dataset
# =========================================================
class CocoDetection(torchvision.datasets.CocoDetection):
    def __init__(
        self,
        image_directory_path,
        annotation_file_path,
        image_processor,
        coco_to_model_cat_id,
        train=True,
        transform=None,
    ):
        super().__init__(image_directory_path, annotation_file_path)
        self.image_processor = image_processor
        self.coco_to_model_cat_id = coco_to_model_cat_id
        self.train = train
        self.transform = transform

    def _clip_bbox_to_image(self, bbox, width, height):
        x, y, w, h = bbox
        x1 = max(0, x)
        y1 = max(0, y)
        x2 = min(width, x + w)
        y2 = min(height, y + h)
        return [x1, y1, max(0, x2 - x1), max(0, y2 - y1)]

    def __getitem__(self, idx):
        image, annotations = super().__getitem__(idx)
        image_id = self.ids[idx]

        image = np.array(image)
        height, width = image.shape[:2]

        cleaned_annotations = []
        for ann in annotations:
            bbox = self._clip_bbox_to_image(ann["bbox"], width, height)
            if bbox[2] <= 1 or bbox[3] <= 1:
                continue
            original_cat_id = ann["category_id"]
            model_cat_id = self.coco_to_model_cat_id[original_cat_id]
            cleaned_annotations.append(
                {
                    "bbox": bbox,
                    "category_id": model_cat_id,
                    "area": bbox[2] * bbox[3],
                    "iscrowd": ann.get("iscrowd", 0),
                }
            )

        annotations = cleaned_annotations
        boxes = [ann["bbox"] for ann in annotations]
        category_ids = [ann["category_id"] for ann in annotations]

        if self.train and self.transform is not None and len(boxes) > 0:
            transformed = self.transform(
                image=image, bboxes=boxes, category_ids=category_ids
            )
            image = transformed["image"]
            boxes = transformed["bboxes"]
            category_ids = transformed["category_ids"]

        annotations = [
            {
                "bbox": list(box),
                "category_id": int(cid),
                "area": float(box[2] * box[3]),
                "iscrowd": 0,
            }
            for box, cid in zip(boxes, category_ids)
            if box[2] > 1 and box[3] > 1
        ]

        target = {"image_id": image_id, "annotations": annotations}
        encoding = self.image_processor(
            images=image, annotations=target, return_tensors="pt"
        )

        pixel_values = encoding["pixel_values"].squeeze(0)
        labels = encoding["labels"][0]

        return pixel_values, labels

In [7]:
# =========================================================
# 6. Build category mappings
# =========================================================
tmp_dataset = torchvision.datasets.CocoDetection(TRAIN_DIRECTORY, TRAIN_ANNOTATION_FILE)
categories = tmp_dataset.coco.cats

original_cat_ids = sorted(categories.keys())
coco_to_model_cat_id = {cat_id: idx for idx, cat_id in enumerate(original_cat_ids)}
model_to_coco_cat_id = {idx: cat_id for cat_id, idx in coco_to_model_cat_id.items()}

id2label = {
    idx: categories[cat_id]["name"] for idx, cat_id in enumerate(original_cat_ids)
}
label2id = {name: idx for idx, name in id2label.items()}

print("Original COCO category ids:", original_cat_ids)
print("Model id2label:", id2label)

loading annotations into memory...
Done (t=0.25s)
creating index...
index created!
Original COCO category ids: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Model id2label: {0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6', 7: '7', 8: '8', 9: '9'}


In [8]:
# =========================================================
# 7. Create datasets
# =========================================================
TRAIN_DATASET = CocoDetection(
    image_directory_path=TRAIN_DIRECTORY,
    annotation_file_path=TRAIN_ANNOTATION_FILE,
    image_processor=image_processor,
    coco_to_model_cat_id=coco_to_model_cat_id,
    train=True,
    transform=train_transform,
)

VAL_DATASET = CocoDetection(
    image_directory_path=VAL_DIRECTORY,
    annotation_file_path=VAL_ANNOTATION_FILE,
    image_processor=image_processor,
    coco_to_model_cat_id=coco_to_model_cat_id,
    train=False,
    transform=None,
)

print("Number of training examples:", len(TRAIN_DATASET))
print("Number of validation examples:", len(VAL_DATASET))

loading annotations into memory...
Done (t=0.28s)
creating index...
index created!
loading annotations into memory...
Done (t=0.20s)
creating index...
index created!
Number of training examples: 30062
Number of validation examples: 3340


In [9]:
# # =========================================================
# # 8. collate_fn
# # =========================================================
def collate_fn(batch):
    pixel_values = [item[0] for item in batch]  # each: [C, H, W]
    labels = [item[1] for item in batch]

    max_h = max(img.shape[1] for img in pixel_values)
    max_w = max(img.shape[2] for img in pixel_values)

    padded_images = []
    pixel_masks = []

    for img in pixel_values:
        c, h, w = img.shape

        padded_img = torch.zeros((c, max_h, max_w), dtype=img.dtype)
        padded_img[:, :h, :w] = img
        padded_images.append(padded_img)

        mask = torch.zeros((max_h, max_w), dtype=torch.long)
        mask[:h, :w] = 1
        pixel_masks.append(mask)

    return {
        "pixel_values": torch.stack(padded_images),  # [B, C, H, W]
        "pixel_mask": torch.stack(pixel_masks),  # [B, H, W]
        "labels": labels,
    }

In [10]:
# =========================================================
# 9. DataLoaders
# =========================================================
TRAIN_DATALOADER = DataLoader(
    TRAIN_DATASET,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True if DEVICE == "cuda" else False,
)

VAL_DATALOADER = DataLoader(
    VAL_DATASET,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True if DEVICE == "cuda" else False,
)

In [11]:
# =========================================================
# 10. Build model
# =========================================================


print("Building model...")

config = DeformableDetrConfig(
    # only use pretrained in the backbone
    use_timm_backbone=True,
    backbone="resnet50d",
    use_pretrained_backbone=True,
    dilation=True,
    # labels
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
    # queries & two-stage
    num_queries=50,
    two_stage=True,
    two_stage_num_proposals=50,
    with_box_refine=True,
    # feature levels
    num_feature_levels=3,
    # encoder/decoder
    encoder_layers=4,
    decoder_layers=4,
    encoder_ffn_dim=1024,
    decoder_ffn_dim=1024,
    d_model=256,
    # dropout
    dropout=0.1,
    attention_dropout=0.1,
    # loss
    auxiliary_loss=True,
    class_cost=2,
    bbox_cost=5,
    giou_cost=2,
    bbox_loss_coefficient=5,
    giou_loss_coefficient=2,
    eos_coefficient=0.02,
)

model = DeformableDetrForObjectDetection(config)
model.to(DEVICE)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {num_params:,}")

if ENABLE_GRADIENT_CHECKPOINTING and hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
    print("Gradient checkpointing enabled.")

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
print("Model ready.")

Building model...
Trainable parameters: 32,312,742
Gradient checkpointing enabled.
Model ready.


In [12]:
# =========================================================
# 11. Freeze backbone
# =========================================================


def set_encoder_decoder_grad(model, requires_grad):
    for name, param in model.named_parameters():
        if "backbone" not in name:
            param.requires_grad = requires_grad
    status = "Unfrozen" if requires_grad else "Frozen"
    print(f"Encoder/Decoder: {status}")


set_encoder_decoder_grad(model, False)

Encoder/Decoder: Frozen


In [13]:
# =========================================================
# 12. Optimizer & Scheduler
# =========================================================
param_dicts = [
    {
        "params": [
            p
            for n, p in model.named_parameters()
            if "backbone" not in n and p.requires_grad
        ]
    },
    {
        "params": [
            p
            for n, p in model.named_parameters()
            if "backbone" in n and p.requires_grad
        ],
        "lr": LR_BACKBONE,
    },
]

optimizer = torch.optim.AdamW(param_dicts, lr=LR, weight_decay=WEIGHT_DECAY)

scheduler = OneCycleLR(
    optimizer,
    max_lr=[LR, LR_BACKBONE],
    steps_per_epoch=len(TRAIN_DATALOADER),
    epochs=NUM_EPOCHS,
    pct_start=0.1,  # Early 10% warmup
    anneal_strategy="cos",
    div_factor=10.0,
    final_div_factor=1000.0,
)

In [14]:
# =========================================================
# 13. Memory cleanup helper
# =========================================================
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [15]:
# =========================================================
# 14. COCO evaluator helper
# =========================================================
def prepare_for_coco_detection(predictions, model_to_coco_cat_id):
    coco_results = []
    for image_id, outputs in predictions.items():
        if len(outputs["boxes"]) == 0:
            continue
        boxes = outputs["boxes"].tolist()
        scores = outputs["scores"].tolist()
        labels = outputs["labels"].tolist()
        for box, score, label in zip(boxes, scores, labels):
            x1, y1, x2, y2 = box
            coco_category_id = model_to_coco_cat_id[int(label)]
            coco_results.append(
                {
                    "image_id": image_id,
                    "category_id": coco_category_id,
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "score": score,
                }
            )
    return coco_results

In [ ]:
# =========================================================
# 15. Train one epoch
# =========================================================
def train_one_epoch(
    model,
    dataloader,
    optimizer,
    scheduler,
    device,
    scaler,
    grad_accum_steps=1,
    use_amp=False,
):
    model.train()
    total_loss = 0.0
    total_loss_dict = {}

    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(dataloader, desc="Training", leave=False)

    for step, batch in enumerate(pbar):
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask = batch["pixel_mask"].to(device)
        labels = [{k: v.to(device) for k, v in t.items()}
                  for t in batch["labels"]]

        with torch.amp.autocast("cuda", enabled=use_amp):
            outputs = model(
                pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels
            )
            loss = outputs.loss
            loss_dict = outputs.loss_dict
            loss_for_backward = loss / grad_accum_steps

        scaler.scale(loss_for_backward).backward()

        if ((step + 1) % grad_accum_steps == 0) or ((step + 1) == len(dataloader)):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),
                                           max_norm=GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        total_loss += loss.item()
        for k, v in loss_dict.items():
            total_loss_dict[k] = total_loss_dict.get(k, 0.0) + v.item()

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / len(dataloader)
    avg_loss_dict = {k: v / len(dataloader)
                     for k, v in total_loss_dict.items()}
    return avg_loss, avg_loss_dict

In [ ]:
# =========================================================
# 16. Validation + COCO mAP
# =========================================================
@torch.no_grad()
def evaluate_with_coco(
    model,
    dataloader,
    dataset,
    device,
    image_processor,
    model_to_coco_cat_id,
    use_amp=False,
):
    model.eval()
    total_loss = 0.0
    total_loss_dict = {}
    evaluator = CocoEvaluator(coco_gt=dataset.coco, iou_types=["bbox"])
    pbar = tqdm(dataloader, desc="Validation", leave=False)

    for batch in pbar:
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask = batch["pixel_mask"].to(device)
        labels = [{k: v.to(device) for k, v in t.items()}
                  for t in batch["labels"]]

        with torch.amp.autocast("cuda", enabled=use_amp):
            outputs_with_loss = model(
                pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels
            )

        loss = outputs_with_loss.loss
        loss_dict = outputs_with_loss.loss_dict
        total_loss += loss.item()
        for k, v in loss_dict.items():
            total_loss_dict[k] = total_loss_dict.get(k, 0.0) + v.item()

        with torch.amp.autocast("cuda", enabled=use_amp):
            outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)

        orig_target_sizes = torch.stack(
            [target["orig_size"] for target in labels], dim=0
        )
        results = image_processor.post_process_object_detection(
            outputs, threshold=0.0, target_sizes=orig_target_sizes
        )

        predictions = {
            target["image_id"].item(): {
                "boxes": output["boxes"].detach().cpu(),
                "scores": output["scores"].detach().cpu(),
                "labels": output["labels"].detach().cpu(),
            }
            for target, output in zip(labels, results)
        }
        predictions = prepare_for_coco_detection(predictions,
                                                 model_to_coco_cat_id)
        evaluator.update(predictions)
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / len(dataloader)
    avg_loss_dict = {k: v / len(dataloader)
                     for k, v in total_loss_dict.items()}

    evaluator.synchronize_between_processes()
    evaluator.accumulate()
    evaluator.summarize()

    coco_stats = evaluator.coco_eval["bbox"].stats
    metrics = {
        "map": float(coco_stats[0]),
        "map_50": float(coco_stats[1]),
        "map_75": float(coco_stats[2]),
    }
    return avg_loss, avg_loss_dict, metrics

In [18]:
# =========================================================
# 17. History
# =========================================================
history = {
    "train_loss": [],
    "val_loss": [],
    "val_map": [],
    "val_map_50": [],
    "val_map_75": [],
}

In [ ]:
# =========================================================
# 18. Main training loop
# =========================================================
best_map = -1.0
best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):
    print(f"\n========== Epoch {epoch + 1}/{NUM_EPOCHS} ==========")

    # Unfreezing strategy
    if epoch == FREEZE_EPOCHS:
        print("Unfreezing encoder/decoder for joint training")
        set_encoder_decoder_grad(model, True)

        # Rebuild the optimizer after unfreezing
        # so that the newly trainable parameters are included
        param_dicts = [
            {
                "params": [
                    p
                    for n, p in model.named_parameters()
                    if "backbone" not in n and p.requires_grad
                ]
            },
            {
                "params": [
                    p
                    for n, p in model.named_parameters()
                    if "backbone" in n and p.requires_grad
                ],
                "lr": LR_BACKBONE,
            },
        ]
        optimizer = torch.optim.AdamW(param_dicts,
                                      lr=LR,
                                      weight_decay=WEIGHT_DECAY)
        scheduler = OneCycleLR(
            optimizer,
            max_lr=[LR, LR_BACKBONE],
            steps_per_epoch=len(TRAIN_DATALOADER),
            epochs=NUM_EPOCHS - FREEZE_EPOCHS,
            pct_start=0.05,
            anneal_strategy="cos",
            div_factor=10.0,
            final_div_factor=1000.0,
        )

    current_lr_main = optimizer.param_groups[0]["lr"]
    current_lr_backbone = optimizer.param_groups[1]["lr"]
    print(f"LR (main):     {current_lr_main:.2e}")
    print(f"LR (backbone): {current_lr_backbone:.2e}")

    train_loss, train_loss_dict = train_one_epoch(
        model=model,
        dataloader=TRAIN_DATALOADER,
        optimizer=optimizer,
        scheduler=scheduler,
        device=DEVICE,
        scaler=scaler,
        grad_accum_steps=GRAD_ACCUM_STEPS,
        use_amp=USE_AMP,
    )

    val_loss, _, val_metrics = evaluate_with_coco(
        model=model,
        dataloader=VAL_DATALOADER,
        dataset=VAL_DATASET,
        device=DEVICE,
        image_processor=image_processor,
        model_to_coco_cat_id=model_to_coco_cat_id,
        use_amp=USE_AMP,
    )

    val_map = val_metrics["map"]
    val_map_50 = val_metrics["map_50"]
    val_map_75 = val_metrics["map_75"]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_map"].append(val_map)
    history["val_map_50"].append(val_map_50)
    history["val_map_75"].append(val_map_75)

    print(f"Train Loss : {train_loss:.5f}")
    print(f"Val   Loss : {val_loss:.5f}")
    print(f"Val mAP    : {val_map:.5f}")
    print(f"Val mAP@50 : {val_map_50:.5f}")
    print(f"Val mAP@75 : {val_map_75:.5f}")

    if val_map > best_map:
        best_map = val_map
        torch.save(model.state_dict(),
                   os.path.join(SAVE_DIR, "best_model_by_map.pth"))
        print(f"Saved best model (mAP={best_map:.5f}) at Epoch {epoch+1}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(),
                   os.path.join(SAVE_DIR, "best_model_by_loss.pth"))
        print(f"Saved model (loss={best_val_loss:.5f}) at Epoch {epoch+1}")

    if (epoch + 1) % 10 == 0:
        torch.save(model.state_dict(),
                   os.path.join(SAVE_DIR, f"model_{epoch+1}.pth"))
        print(f"Checkpoint saved at Epoch {epoch+1}")

    cleanup_memory()

print("\nTraining finished.")
print("Best mAP:", best_map)
print("Best val loss:", best_val_loss)

In [ ]:
# =========================================================
# 19. Plot training history
# =========================================================
def plot_training_history(history):
    epochs = list(range(1, len(history["train_loss"]) + 1))

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], marker="o", label="Train Loss")
    plt.plot(epochs, history["val_loss"], marker="o", label="Val Loss")
    plt.axvline(
        x=FREEZE_EPOCHS, color="gray",
        linestyle="--",
        label="Unfreeze encoder/decoder"
    )
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training / Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["val_map"], marker="o", label="mAP")
    plt.plot(epochs, history["val_map_50"], marker="o", label="mAP@50")
    plt.plot(epochs, history["val_map_75"], marker="o", label="mAP@75")
    plt.axvline(
        x=FREEZE_EPOCHS, color="gray",
        linestyle="--",
        label="Unfreeze encoder/decoder"
    )
    plt.xlabel("Epoch")
    plt.ylabel("mAP")
    plt.title("Validation mAP")
    plt.legend()
    plt.grid(True)
    plt.show()


plot_training_history(history)

In [ ]:
# =========================================================
# 20. Build model again but loss weigth change
# =========================================================

print("Building model...")

config = DeformableDetrConfig(
    # only use pretrained in the backbone
    use_timm_backbone=True,
    backbone="resnet50d",
    use_pretrained_backbone=True,
    dilation=True,
    # labels
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
    # queries & two-stage
    num_queries=50,
    two_stage=True,
    two_stage_num_proposals=50,
    with_box_refine=True,
    # feature levels
    num_feature_levels=3,
    # encoder/decoder
    encoder_layers=4,
    decoder_layers=4,
    encoder_ffn_dim=1024,
    decoder_ffn_dim=1024,
    d_model=256,
    # dropout
    dropout=0.1,
    attention_dropout=0.1,
    # loss
    auxiliary_loss=True,
    class_cost=2,
    bbox_cost=5,
    giou_cost=5,  # 2 to 5
    bbox_loss_coefficient=5,
    giou_loss_coefficient=5,  # 2 to 5
    eos_coefficient=0.05,  # 0.02 to 0.05
)

model = DeformableDetrForObjectDetection(config)
model.to(DEVICE)

if ENABLE_GRADIENT_CHECKPOINTING and hasattr(model,
                                             "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
    print("Gradient checkpointing enabled.")

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
print("Model ready.")

In [ ]:
# =========================================================
# 21. Fine-tune for bbox head to improve mAP@75
# =========================================================

set_encoder_decoder_grad(model, True)

# load the best checkpoint
model.load_state_dict(
    torch.load(os.path.join(SAVE_DIR, "best_model_by_map.pth"),
               map_location=DEVICE)
)
model.to(DEVICE)
print("Loaded best model for fine-tuning")

# unfreeze bbox_embed and reference_points
for name, param in model.named_parameters():
    if any(k in name for k in ["bbox_embed", "reference_points"]):
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Fine-tune trainable params: {trainable:,}")

# Fine-tune details
FT_EPOCHS = 10
FT_LR = 1e-5
FT_LR_MIN = 1e-7

optimizer_ft = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=FT_LR,
    weight_decay=1e-4
)
scheduler_ft = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_ft, T_max=FT_EPOCHS, eta_min=FT_LR_MIN
)
scaler_ft = torch.amp.GradScaler("cuda", enabled=USE_AMP)

ft_history = {"train_loss": [], "val_map": [],
              "val_map_50": [], "val_map_75": []}

# from the best mAP (Remember changing to real number)
best_ft_map = best_map
print(f"Starting fine-tune from mAP = {best_map:.5f}")

In [ ]:
# =========================================================
# 22. Fine-tune Loop for detailed element
# =========================================================
for epoch in range(FT_EPOCHS):
    print(f"\n=== Fine-tune Epoch {epoch+1}/{FT_EPOCHS} ===")

    #  Train
    model.train()
    total_loss = 0.0
    optimizer_ft.zero_grad(set_to_none=True)
    pbar = tqdm(TRAIN_DATALOADER, desc="FT Train", leave=False)

    for step, batch in enumerate(pbar):
        pixel_values = batch["pixel_values"].to(DEVICE)
        pixel_mask = batch["pixel_mask"].to(DEVICE)
        labels = [{k: v.to(DEVICE) for k, v in t.items()}
                  for t in batch["labels"]]

        with torch.amp.autocast("cuda", enabled=USE_AMP):
            outputs = model(
                pixel_values=pixel_values,
                pixel_mask=pixel_mask,
                labels=labels
            )
            loss = outputs.loss / GRAD_ACCUM_STEPS

        scaler_ft.scale(loss).backward()

        if ((step + 1) % GRAD_ACCUM_STEPS == 0) or (
            (step + 1) == len(TRAIN_DATALOADER)
        ):
            scaler_ft.unscale_(optimizer_ft)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler_ft.step(optimizer_ft)
            scaler_ft.update()
            optimizer_ft.zero_grad(set_to_none=True)

        total_loss += outputs.loss.item()
        pbar.set_postfix(loss=f"{outputs.loss.item():.4f}")

    avg_loss = total_loss / len(TRAIN_DATALOADER)
    scheduler_ft.step()

    # Validate
    _, _, val_metrics = evaluate_with_coco(
        model=model,
        dataloader=VAL_DATALOADER,
        dataset=VAL_DATASET,
        device=DEVICE,
        image_processor=image_processor,
        model_to_coco_cat_id=model_to_coco_cat_id,
        use_amp=USE_AMP,
    )

    ft_history["train_loss"].append(avg_loss)
    ft_history["val_map"].append(val_metrics["map"])
    ft_history["val_map_50"].append(val_metrics["map_50"])
    ft_history["val_map_75"].append(val_metrics["map_75"])

    print(f"Train Loss : {avg_loss:.5f}")
    print(f"Val mAP    : {val_metrics['map']:.5f}")
    print(f"Val mAP@50 : {val_metrics['map_50']:.5f}")
    print(f"Val mAP@75 : {val_metrics['map_75']:.5f}")

    if val_metrics["map"] > best_ft_map:
        best_ft_map = val_metrics["map"]
        torch.save(
            model.state_dict(),
            os.path.join(SAVE_DIR, "best_model_map.pth")
        )
        print(f"New best mAP = {best_ft_map:.5f}, saved!")

    cleanup_memory()

print(f"\nFine-tune done. Best mAP = {best_ft_map:.5f}")